# [PROJECT TITLE]
**DAI Mission — Data & AI in Economics | TU Dortmund**

---

## Team

| Name | Role |
|------|------|
| Firstname Lastname | Lead / Causal Inference |
| Firstname Lastname | Supervised Learning |
| Firstname Lastname *(optional)* | Unsupervised / Generative |

> **Note:** No student IDs in this file. Submit IDs separately via the course system (Moodle).

---

**LLM Assistance Disclosure:**  
*[Required if applicable] We used [tool name] to [brief description, e.g., debug DoWhy syntax].
All analysis, interpretation, and conclusions are our own.*

---

> **Repo naming:** `dai-mission-group-X` (replace X with your group letter)  
> **Submission:** notebook must be committed in a **fully executed state** (all outputs visible).

## Research Question

*[TEMPLATE] Write your one-sentence research question here — what causal or predictive question
are you answering, and with what data?*

---

**Toy example used throughout this template** *(replace everything marked [EXAMPLE] with your own analysis)*:

Does participating in a job-training program causally raise wages? We study this using synthetic
labour-market data (n = 2 000 workers), where selection into training depends on distance to the
training centre (instrument), age, and years of education.

In [ ]:
# [EXAMPLE — keep matplotlib.use('Agg'); update other imports for your own analysis]
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — required for headless CI execution

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_curve, auc, silhouette_score
from sklearn.preprocessing import StandardScaler

from dowhy import CausalModel

np.random.seed(42)
print('All imports successful')

---
## Work Plan

**[TEMPLATE — required]** Map each team member to their section. This table is the basis for
individual accountability during the oral exam: every member must be prepared to answer
questions about their own designated area.

| Section | Responsible member | Main tasks |
|---------|-------------------|------------|
| §1 Research Question & Data | Firstname Lastname | data sourcing, cleaning, variable table |
| §2 Causal Inference | Firstname Lastname | DAG design, DoWhy implementation, refutation |
| §3 Supervised Learning | Firstname Lastname | model selection, training, evaluation |
| §4 Unsupervised / Generative | Firstname Lastname | method choice, implementation, visualisation |
| §5 Synthesis & Communication | all members | cross-method narrative, conclusion, notebook readability |

**Shared tasks:** *[e.g., research question definition, literature review, presentation design]*

> Note: The same notebook serves as both the proposal and the final submission.

---
## Section 1 — Research Question & Data

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] Question is specific, answerable, and economically motivated
- [ ] Data source is clearly identified and appropriate for the question
- [ ] Variables are described with types and roles (feature / target / instrument)
- [ ] Data quality issues are acknowledged and handled (missings, bias, outliers)

---
*Replace the toy data generation below with your own data loading (`pd.read_csv(...)`, API call, etc.).*

In [ ]:
# [EXAMPLE — replace with your data loading]
# Synthetic labour-market data: does a training programme raise wages?
n = 2_000

age       = np.random.randint(22, 60, n)
education = np.random.randint(10, 18, n)         # years of schooling
distance  = np.random.uniform(1, 50, n)          # km to nearest training centre (instrument)

# Treatment: more likely to participate if close, younger, more educated
log_odds = -0.05 * distance + 0.03 * education - 0.01 * age + np.random.normal(0, 0.5, n)
prob     = 1 / (1 + np.exp(-log_odds))
training = (np.random.uniform(0, 1, n) < prob).astype(int)

# Outcome: log wage depends on training AND confounders
log_wage = 0.4 * training + 0.05 * education + 0.008 * age + np.random.normal(0, 0.3, n)

df = pd.DataFrame({
    'age': age, 'education': education, 'distance': distance,
    'training': training, 'log_wage': log_wage
})

print(f'Dataset: {len(df):,} observations | {df["training"].mean():.1%} participated in training')
df.describe().round(2)

**[TEMPLATE] Describe your variables:**

| Variable | Type | Role | Description |
|----------|------|------|-------------|
| `age` | integer | confounder | Worker age in years (22–59) |
| `education` | integer | confounder | Years of schooling (10–17) |
| `distance` | float | instrument | Distance (km) to nearest training centre |
| `training` | binary | **treatment** | 1 = participated in programme, 0 = did not |
| `log_wage` | float | **outcome** | Log monthly wage |

**Data quality notes:**  
*[TEMPLATE: Describe any missing values, outliers, or sampling bias in your real dataset.
Example: "8% of wage records are missing; we impute with group median."]*

---
## Section 2 — Causal Inference Block

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] A causal graph (DAG) is constructed and the assumed relationships are justified
- [ ] Identification strategy is appropriate (backdoor / IV / propensity score)
- [ ] Estimation is implemented correctly using DoWhy (or equivalent)
- [ ] At least one refutation test is run and its result is interpreted

---
*This example uses backdoor adjustment (linear regression). Replace with the strategy appropriate for your research question.*

In [ ]:
# [EXAMPLE — replace with your causal graph]
# Draw the DAG using networkx — no graphviz system package required
G = nx.DiGraph()
G.add_edges_from([
    ('age',       'training'),
    ('education', 'training'),
    ('distance',  'training'),   # instrument: affects treatment but not outcome directly
    ('age',       'log_wage'),
    ('education', 'log_wage'),
    ('training',  'log_wage'),   # causal effect of interest
])

pos = {
    'distance':  (0.0,  0.0),
    'age':       (1.2,  1.2),
    'education': (1.2, -1.2),
    'training':  (2.8,  0.0),
    'log_wage':  (4.4,  0.0),
}
node_colors = [
    '#AED6F1' if n == 'distance' else
    '#FDEBD0' if n in ('age', 'education') else
    '#F9E79F' if n == 'training' else
    '#A9DFBF'   # log_wage
    for n in G.nodes()
]

fig, ax = plt.subplots(figsize=(9, 4))
nx.draw_networkx(G, pos=pos, ax=ax, node_color=node_colors,
                 node_size=2500, font_size=9, arrows=True,
                 arrowsize=20, edge_color='dimgray', width=1.5)
ax.set_title('Causal DAG: Job-Training Programme \u2192 Log Wage', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('dag.png', dpi=100, bbox_inches='tight')
plt.close()
print('DAG saved \u2192 dag.png')
print('Blue=instrument | Orange=confounders | Yellow=treatment | Green=outcome')

In [ ]:
# [EXAMPLE — replace variable names and graph string for your study]
dot_graph = (
    'digraph {'
    ' age -> training; education -> training; distance -> training;'
    ' age -> log_wage; education -> log_wage; training -> log_wage;'
    '}'
)

model = CausalModel(
    data=df,
    treatment='training',
    outcome='log_wage',
    graph=dot_graph,
)

identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

In [ ]:
# [EXAMPLE — backdoor.linear_regression is the simplest estimator; choose what fits your design]
causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name='backdoor.linear_regression',
    target_units='ate',
)
ate = causal_estimate.value
print(causal_estimate)
print(f'\nEstimated Average Treatment Effect (ATE): {ate:.4f}')
print(
    f'Interpretation: training participation is associated with a '
    f'{ate:+.2%} change in log wages, holding age and education constant.'
)

In [ ]:
# [EXAMPLE — random_common_cause refuter]
# Adds a random variable as a common cause; a robust estimate should not change substantially
refutation = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name='random_common_cause',
    num_simulations=5,
)
print(refutation)
print('\n[TEMPLATE] Interpret the refutation: did the estimate remain stable?')
print('If the new estimate is close to the original, the result is robust to this check.')

---
## Section 3 — Supervised Learning Block

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] Model choice is justified relative to the prediction task
- [ ] Train/test split and cross-validation are used correctly
- [ ] An appropriate metric is reported and interpreted (RMSE, AUC, accuracy, …)
- [ ] Results are compared to a baseline or alternative model

---
*This example predicts training participation (binary classification) from worker characteristics.
Replace with your own prediction task — classification or regression.*

In [ ]:
# [EXAMPLE — replace X, y, and model choices for your task]
# Prediction task: who participates in training? (binary classification)
X = df[['age', 'education', 'distance']]
y = df['training']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline: logistic regression
lr = LogisticRegression(max_iter=500)
lr.fit(X_train, y_train)
lr_proba = lr.predict_proba(X_test)[:, 1]

# Alternative: random forest
rf = RandomForestClassifier(n_estimators=100, n_jobs=1, random_state=42)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

# ROC curves
fig, ax = plt.subplots(figsize=(6, 5))
for label, proba in [('Logistic Regression (baseline)', lr_proba),
                     ('Random Forest', rf_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_score   = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{label} (AUC = {auc_score:.3f})', linewidth=1.8)

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves: Predicting Training Participation')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=100, bbox_inches='tight')
plt.close()
print('ROC curves saved \u2192 roc_curves.png')

In [ ]:
# [EXAMPLE — 5-fold cross-validation on the full dataset]
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='roc_auc', n_jobs=1)
print(f'Random Forest | 5-fold CV AUC: {cv_scores.mean():.3f} \u00b1 {cv_scores.std():.3f}')
print(f'Per-fold:      {[round(s, 3) for s in cv_scores]}')

**[TEMPLATE] Interpret your supervised learning results:**

- **Why this model?** *Justify your model choice for your specific prediction task and data type.*
- **Key metric:** *Report and interpret your chosen metric — why is it appropriate for this task?*
- **Baseline comparison:** *How does your best model compare to the simpler baseline?*
- **Limitations:** *Any overfitting concerns? Class imbalance? Feature leakage risks?*

---
## Section 4 — Unsupervised / Generative Block

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] Method choice is justified relative to the structure of the data or task
- [ ] Implementation is correct (k selection, linkage choice, latent dim, …)
- [ ] Output is evaluated with an appropriate measure (silhouette, reconstruction loss, …)
- [ ] Findings are visualised and interpreted in domain terms

---
*This example clusters workers into latent types using K-Means + PCA.
Replace with your method: hierarchical clustering, VAE, GAN, topic model, etc.*

In [ ]:
# [EXAMPLE — K-Means on standardised features, visualised in PCA space]
features = df[['age', 'education', 'log_wage']]
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(features)

# PCA for 2-D visualisation
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Elbow curve: choose k
ks       = range(2, 7)
inertias = [KMeans(n_clusters=k, random_state=42, n_init='auto').fit(X_scaled).inertia_ for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('k (number of clusters)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_title('Elbow Curve')
axes[0].grid(alpha=0.3)

k_chosen = 3
km       = KMeans(n_clusters=k_chosen, random_state=42, n_init='auto')
labels   = km.fit_predict(X_scaled)
sil      = silhouette_score(X_scaled, labels)
print(f'K={k_chosen} | Silhouette score: {sil:.3f} (range: -1 to +1; higher is better)')

sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=10, alpha=0.6)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
axes[1].set_title(f'K-Means Clusters (k={k_chosen}) in PCA Space')
plt.colorbar(sc, ax=axes[1], label='Cluster')
plt.tight_layout()
plt.savefig('clusters.png', dpi=100, bbox_inches='tight')
plt.close()
print('Cluster plot saved \u2192 clusters.png')

In [ ]:
# [EXAMPLE — cluster centroids back in original feature scale]
df['cluster'] = labels
centroids_orig = scaler.inverse_transform(km.cluster_centers_)
centroid_df = pd.DataFrame(
    centroids_orig,
    columns=['age', 'education', 'log_wage'],
    index=[f'Cluster {i}' for i in range(k_chosen)]
)
centroid_df['n_workers']     = df.groupby('cluster').size().values
centroid_df['training_rate'] = df.groupby('cluster')['training'].mean().values
print('Cluster centroids (original scale):')
print(centroid_df.round(2))

**[TEMPLATE] Interpret your unsupervised results:**

- **Why this method?** *Justify in relation to the data structure and research question.*
- **How was k (or another hyperparameter) chosen?** *Describe the trade-off you observed.*
- **What do the clusters mean economically?** *Describe each cluster in plain language: who are these workers?*
- **Limitations:** *Sensitivity to initialisation? Non-spherical clusters? Curse of dimensionality?*

---
## Section 5 — Synthesis & Communication

**[TEMPLATE] Rubric checklist (4 pts total):**
- [ ] The three method blocks are connected — each result informs the next
- [ ] Conclusions directly answer the research question
- [ ] Limitations and potential confounders are honestly discussed
- [ ] Notebook is readable: clear markdown narrative, labelled plots, no dead code

---
*Replace the toy narrative below with your own synthesis.*

### What causal inference revealed

*[TEMPLATE] Summarise your ATE estimate and what it implies for the research question.
Was the effect statistically and economically meaningful?*

**Toy example:** Backdoor adjustment estimates an ATE of approximately +0.40 log-wage units,
suggesting the training programme substantially raises wages after controlling for age and
education. The random-common-cause refuter confirms the estimate is robust to an added
spurious confounder.

---

### What supervised learning revealed

*[TEMPLATE] What does the predictive model tell you about who participates?
Which features matter most? How does this connect to the causal story?*

**Toy example:** Distance to the training centre is the strongest predictor of participation
(as designed — it is the instrument). The Random Forest outperforms logistic regression
(AUC \u2248 0.85 vs \u2248 0.78), suggesting nonlinear selection effects.

---

### What clustering revealed

*[TEMPLATE] How do the clusters relate to your treatment and outcome?
Do certain worker types benefit more from training?*

**Toy example:** Three worker types emerge: (0) young/low-education workers with the highest
training rate; (1) prime-age/highly-educated workers with the highest baseline wage;
(2) older/medium-education workers with the lowest training rate.

---

### Limitations & honest discussion

*[TEMPLATE — required for full marks] Discuss what your analysis cannot establish.
Examples: unmeasured confounders, external validity, distributional assumptions,
model misspecification, data representativeness.*

---

### Conclusion

*[TEMPLATE] 2–3 sentences directly answering the research question stated in Section 1.*

In [ ]:
# [EXAMPLE — 3-panel summary figure connecting all three method blocks]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: ATE point estimate
axes[0].bar(['Training\nEffect'], [ate], color='#A9DFBF', width=0.4)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_ylabel('Estimated ATE (log wage)')
axes[0].set_title('\u00a72 Causal Effect\n(backdoor adjustment)')
axes[0].grid(axis='y', alpha=0.3)

# Panel 2: ROC curves
for label, proba, color in [('LogReg', lr_proba, '#AED6F1'),
                              ('RF',    rf_proba, '#F9E79F')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[1].plot(fpr, tpr, label=f'{label} (AUC={auc(fpr, tpr):.2f})',
                 color=color, linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('\u00a73 Supervised Learning\n(ROC curves)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# Panel 3: cluster scatter
sc = axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=8, alpha=0.5)
axes[2].set_xlabel('PC1')
axes[2].set_ylabel('PC2')
axes[2].set_title(f'\u00a74 Unsupervised\n(K={k_chosen} worker types)')
plt.colorbar(sc, ax=axes[2], label='Cluster')

plt.suptitle('DAI Mission \u2014 Summary of Results', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('summary_figure.png', dpi=100, bbox_inches='tight')
plt.close()
print('Summary figure saved \u2192 summary_figure.png')

---
## References

*List all sources in APA format. Include data sources, key papers, and code libraries. Replace the examples below.*

Sharma, A., & Kiciman, E. (2020). DoWhy: An end-to-end library for causal inference. *arXiv:2011.04216*.

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*, 2825\u20132830.

Author, A. A., & Author, B. B. (Year). Title of article. *Journal Name*, *volume*(issue), pages. https://doi.org/...

Dataset: [Name of dataset]. Retrieved from [URL]. Accessed [date].